<a href="https://colab.research.google.com/github/lucassouza147770-art/MLCB_LG_2026/blob/main/AULA_07/lab01_aula07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
#EXERCÍCIO 1: Regressão Logística (Chatbot Bancário)
#Foco do Modelo: Regressão Logística (LogisticRegression)
#Objetivo: Implementar o pipeline base completo com classificação probabilística e extração de valores monetários.

# =====================================================================
# LAB 1: NLU BANCÁRIO COM REGRESSÃO LOGÍSTICA
# =====================================================================

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

dados_treino = [
    ("Quero transferir dinheiro", "transferencia"),
    ("Preciso fazer um pix", "transferencia"),
    ("Quero consultar meu saldo", "saldo"),
    ("Quanto tenho na conta", "saldo"),
    ("Quero pagar uma conta", "pagamento"),
    ("Preciso pagar meu boleto", "pagamento"),
    ("Quero falar sobre meu cartão", "cartao"),
    ("Meu cartão não funciona", "cartao"),
    ("Quero saber o horário da agência", "fora_escopo")
]

stopwords = ["o", "a", "e", "do", "da", "de", "para", "meu", "minha"]

# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    tokens_filtrados = [token for token in tokens if token not in stopwords]

    return " ".join(tokens_filtrados)

X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]

# 2. VETORIZAÇÃO E TREINAMENTO
vectorizer = TfidfVectorizer()
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)

modelo = LogisticRegression()
modelo.fit(X_vetorizado, y_treino)

# 3. EXTRAÇÃO DE ENTIDADE
def extrair_valor_reais(texto_original):
    match = re.search(r'(\d+)\s*reais', texto_original, re.IGNORECASE)

    if match:
        return float(match.group(1))
    else:
        return None

# 4. PIPELINE NLU
def processar_nlu(mensagem_usuario, threshold=0.55):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo.predict_proba(vetor_input)[0]

    maior_confianca = max(probas)
    intencao_prevista = modelo.classes_[probas.argmax()]

    valor = extrair_valor_reais(mensagem_usuario)

    if maior_confianca < threshold or intencao_prevista == "fora_escopo":
        return {
            "status": "FALLBACK",
            "confianca": round(maior_confianca, 2)
        }
    else:
        return {
            "status": "SUCESSO",
            "intencao": intencao_prevista,
            "confianca": round(maior_confianca, 2),
            "valor": valor
        }

# TESTE 1
if __name__ == "__main__":
    print(processar_nlu("Quero fazer um pix de 200 reais"))

{'status': 'FALLBACK', 'confianca': np.float64(0.37)}
